# 🚇 Metro Railway Document Classification with BERT

This notebook implements a state-of-the-art document classification system for metro railway documents using BERT.

## 📋 What this notebook does:
1. Sets up the complete BERT classification environment
2. Trains a model to classify documents into 6 metro railway categories
3. Tests the model performance
4. Saves the trained model for download

## ⏱️ Expected time: 30-60 minutes

---

## 🔧 Step 1: Environment Setup

First, let's verify GPU access and install required packages.

In [ ]:
import torch
import sys

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("✅ GPU ready for training!")
else:
    print("⚠️  GPU not available - training will be slower on CPU")
    print("💡 Go to Runtime → Change runtime type → Select GPU")

Python version: 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]
PyTorch version: 2.8.0+cu126
CUDA available: True
GPU device: Tesla T4
GPU memory: 15.8 GB
✅ GPU ready for training!


In [ ]:
# Install required packages
!pip install transformers[torch] datasets
!pip install scikit-learn pandas numpy matplotlib seaborn
!pip install PyPDF2 pytesseract pillow
!pip install accelerate  # For faster training

# Install system packages for OCR
!apt-get update -qq
!apt-get install -y tesseract-ocr -qq

print("✅ All packages installed successfully!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
✅ All packages installed successfully!


In [ ]:
# Verify all imports work
try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
    import pandas as pd
    import numpy as np
    import torch
    from sklearn.metrics import accuracy_score, classification_report
    import matplotlib.pyplot as plt
    import warnings
    warnings.filterwarnings('ignore')
    print("✅ All imports successful!")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Please run the previous cell to install packages")

✅ All imports successful!


## 🤖 Step 2: Enhanced BERT Document Classifier

Now let's create our enhanced BERT document classifier with all the features.

In [ ]:
# Enhanced BERT Document Classifier - Complete Implementation
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, precision_recall_fscore_support
)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import logging
import os
import json
from datetime import datetime
from pathlib import Path

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class MetroDocumentDataset(Dataset):
    """Enhanced dataset for metro railway documents"""

    def __init__(self, texts, labels, tokenizer, max_length=512, label_to_id=None):
        self.texts = [str(text) for text in texts]
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.label_to_id = label_to_id or {}

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        if isinstance(label, str) and self.label_to_id:
            label = self.label_to_id[label]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }


class EnhancedBERTDocumentClassifier:
    """Enhanced BERT classifier for metro railway documents"""

    def __init__(self, model_name='bert-base-uncased', num_labels=6, max_length=512):
        self.model_name = model_name
        self.num_labels = num_labels
        self.max_length = max_length
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = None
        self.trainer = None
        self.training_history = {}
        # Ensure self.device is defined in __init__
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


        self.categories = [
            'Technical & Engineering Documents',
            'Passenger & Public-Facing Documents',
            'Financial & Procurement Documents',
            'Human Resources & Administrative Documents',
            'Safety, Security & Regulatory Documents',
            'Strategic & Project Management Documents'
        ]

        self.label_to_id = {label: idx for idx, label in enumerate(self.categories)}
        self.id_to_label = {idx: label for idx, label in enumerate(self.categories)}

        logger.info(f"Initialized classifier with {self.num_labels} categories")

    def prepare_enhanced_sample_data(self):
        """Create enhanced sample training data"""
        sample_docs = {
    'Technical & Engineering Documents': [
        "Architectural blueprints and structural engineering designs for metro station construction with load-bearing calculations and seismic analysis requirements",
        "Electrical schematic diagrams for 25kV overhead catenary system power distribution network installation with substation specifications",
        "Track geometry inspection report documenting rail alignment deviations, gauge measurements, and corrective maintenance requirements",
        "Automated train control signaling system specifications including CBTC communication protocols, ATP safety interlocks, and ATO functionality",
        "Rolling stock technical manual covering traction motors, regenerative braking systems, HVAC equipment maintenance, and door control mechanisms",
        "Civil engineering soil analysis report for tunnel boring machine operation, ground settlement monitoring, and foundation stability assessment",
        "Mechanical ventilation system design specifications for underground station air circulation, smoke extraction, and emergency exhaust capabilities",
        "Platform screen door installation guidelines with electrical interfacing, passenger safety protocols, and emergency override procedures",
        "Signal maintenance procedure documenting LED aspect replacement, cable testing, and communication system troubleshooting methods",
        "Power supply system technical specifications including transformer ratings, protective relay settings, and fault detection mechanisms",
        "Design review documentation for the new elevated track section, focusing on material stress tolerance and expansion joint placement",
        "Geotechnical survey results for a tunnel expansion project, including rock stability analysis and groundwater management plans",
        "Maintenance log for automated fare collection gates, detailing software updates, hardware repairs, and calibration records",
        "System integration protocol for connecting a new security surveillance network with the existing IT infrastructure and data center",
        "Technical drawing for a traction substation layout, including the positioning of rectifiers, transformers, and switchgear for high-voltage power conversion",
        "Failure analysis report for a train's auxiliary power supply unit, identifying the root cause of the component failure and recommending a design change",
        "Installation manual for a fiber optic communication network, specifying cable routing, splicing procedures, and signal testing protocols",
        "Detailed plan for the construction of an underground pedestrian walkway, including structural supports, lighting design, and drainage systems",
        "Blueprint for the extension of the metro depot, showing the layout of new maintenance tracks, inspection pits, and workshop facilities",
        "Specifications for new signaling equipment, including a list of approved vendors and compliance requirements for train control system integration",
        "Software update procedure for the central dispatch system, detailing the rollback plan, testing protocols, and version control",
        "Report on the structural integrity of a viaduct, based on a recent drone inspection and material fatigue analysis",
        "Specifications for specialized welding equipment used for track repair, including safety features and operational procedures",
        "Hydraulic system maintenance manual for train braking systems, detailing fluid replacement schedules and pressure testing procedures",
        "Electrical power consumption report for an entire metro line, analyzing energy usage by station and train operation patterns",
        "Detailed schematics for the station's fire suppression system, including sprinkler head placement and control panel wiring diagrams",
        "Technical proposal for a smart lighting system to be installed in all metro tunnels, including energy-saving features and remote monitoring capabilities",
        "Report on rail wear and tear analysis, recommending a schedule for rail grinding and replacement based on operational data",
        "Design document for the installation of new communication towers along the metro line, including tower specifications and radio frequency planning",
        "Project plan for the upgrading of all station escalators to new, more energy-efficient models with safety sensor integrations"
    ],
    'Passenger & Public-Facing Documents': [
        "Metro system route map displaying all lines, stations, interchange connections, and accessibility features for passenger navigation assistance",
        "Customer service complaint form documenting passenger feedback regarding train delays, overcrowding, and service quality concerns",
        "Fare structure notification announcing revised tariff rates, distance-based pricing, concession categories, and smart card benefits",
        "Accessibility guide for disabled passengers outlining elevator locations, tactile guidance systems, and staff assistance procedures",
        "Public announcement script for service interruptions, alternative transportation options, estimated restoration times, and passenger safety instructions",
        "Smart card user manual explaining top-up procedures, balance checking methods, automatic fare collection system, and card replacement process",
        "Station facility information including restroom locations, commercial areas, parking availability, bicycle storage, and connecting bus services",
        "Emergency passenger communication protocol for evacuation procedures, safety instructions during incidents, and coordination with emergency services",
        "Mobile application user guide covering real-time train information, journey planning, fare calculation, and customer feedback submission",
        "Passenger satisfaction survey questionnaire collecting feedback on service quality, cleanliness, punctuality, and overall travel experience",
        "Press release announcing the opening of a new metro line and its key features for public awareness and media engagement",
        "Public safety poster detailing emergency contact numbers, fire extinguisher locations, and safe boarding procedures at the station",
        "Lost and found claim form for passengers to report missing items and provide contact information for retrieval",
        "User guide for the new ticketing kiosks, explaining how to select a destination, pay with a credit card, and collect a ticket",
        "Community engagement report for the new metro line project, summarizing public feedback and addressing local concerns",
        "Passenger rights charter outlining the standards of service, compensation policies for delays, and complaint resolution procedures",
        "Guide to local attractions and public transport links available from each metro station, with opening hours and ticket information",
        "Promotional leaflet for a new monthly travel pass, highlighting the cost savings and convenience for daily commuters",
        "Code of conduct for passengers, outlining rules on eating, drinking, and appropriate behavior within metro premises",
        "Annual report on customer satisfaction metrics, including ratings for staff helpfulness, station cleanliness, and ticket accessibility",
        "Detailed station evacuation map showing all exits, emergency exits, and assembly points for public reference",
        "Instructions for how to use the station's public WiFi network, including login procedures and data usage policies",
        "List of prohibited items on the metro, with illustrations and safety warnings for passengers",
        "Public notice about construction work near a metro station, advising passengers of temporary closures and alternative routes",
        "Feedback form for accessibility features, allowing disabled passengers to provide specific comments on elevator and ramp availability",
        "Service disruption alert message for a line closure, providing a link to a temporary bus shuttle schedule",
        "Map showing all designated smoking areas and waste disposal locations within the metro system",
        "Passenger guide to using the escalators and elevators safely, with specific warnings for children and luggage",
        "Promotional brochure for a new metro line extension, showcasing a virtual tour of the future stations and key landmarks",
        "Public bulletin regarding changes to operating hours during a public holiday, advising passengers on the last train times"
    ],
    'Tenders & Procurement': [
        "Annual financial statement with comprehensive revenue analysis, operational expenditure breakdown, capital investment summary, and profitability metrics",
        "Request for proposal document for metro car procurement specifying technical requirements, delivery schedules, evaluation criteria, and contract terms",
        "Contractor payment voucher for civil works completion with milestone verification, quality compliance certification, and performance bond details",
        "Vendor service level agreement defining performance metrics, availability targets, penalty clauses, maintenance responsibility scope, and escalation procedures",
        "Budget allocation report for infrastructure development, rolling stock acquisition, operational cost projections, and contingency fund management",
        "Audit findings report highlighting financial irregularities, compliance gaps, internal control weaknesses, and recommended corrective actions",
        "Tender evaluation committee report comparing bidder proposals on technical merit, commercial viability, past performance, and financial capability",
        "Procurement policy document establishing vendor qualification criteria, bidding procedures, contract award process, and conflict of interest guidelines",
        "Cost-benefit analysis for metro line extension project including construction costs, ridership projections, revenue forecasts, and payback period",
        "Invoice processing workflow documentation covering approval hierarchy, payment terms, tax calculations, and dispute resolution procedures",
        "Quarterly financial report with detailed balance sheet, income statement, and cash flow analysis for investor presentation",
        "Purchase order for signal equipment, detailing the quantity, unit price, delivery date, and total cost of the components",
        "Memorandum of Understanding with a private firm for the co-development of a commercial complex at a metro station site",
        "Request for quotation for station furniture, including specifications for benches, waste bins, and information kiosks",
        "Treasury report summarizing daily cash balances, investment portfolio performance, and short-term liquidity projections",
        "Disbursement request for the payment of a contractor's final invoice, with a checklist for verification and approval",
        "Grant proposal application for government funding of a new metro line, outlining the project's financial needs and economic benefits",
        "Annual budget plan for the maintenance and repair department, with a breakdown of labor costs, spare parts inventory, and external service contracts",
        "Contract amendment for a construction project, detailing changes to the project scope, timeline, and associated costs",
        "Due diligence report on a potential acquisition of a land parcel for a new station, including property valuation and legal title review",
        "Statement of work for a software development project for the metro's ticketing system, including deliverables, milestones, and payment schedule",
        "Procurement strategy document for a major project, outlining the sourcing plan, risk mitigation, and supplier management policies",
        "Cost estimation spreadsheet for the construction of a new tunnel section, including material costs, labor rates, and contingency allowances",
        "Bid submission form from a vendor for a rolling stock maintenance contract, detailing their technical and commercial offer",
        "Financial projections for a new metro line, including estimated revenue from fares, advertising, and retail spaces",
        "Internal audit report on the procurement process, highlighting areas of non-compliance and making recommendations for process improvement",
        "Budgetary approval memo for the purchase of new security cameras for all stations, outlining the justification and fund source",
        "Investment analysis report for a new technology adoption, detailing the return on investment and potential cost savings over a five-year period",
        "Credit rating report for the metro company, assessing its financial stability and ability to meet its debt obligations",
        "Invoice from a supplier for electrical components, listing the quantity, price per unit, and total amount due"
    ],
    'Human Resources & Administrative Documents': [
        "Employee recruitment policy defining selection criteria, interview process, background verification, probationary period requirements, and onboarding procedures",
        "Staff training program curriculum for technical skills development, safety procedures, customer service excellence, and professional certification requirements",
        "Performance evaluation report assessing employee productivity, goal achievement, competency development, career progression, and training needs",
        "Organizational restructuring memo announcing departmental changes, reporting relationships, role redefinitions, and transition timelines",
        "Salary revision notification implementing pay scale adjustments, allowance modifications, incentive structure changes, and benefit enhancements",
        "Board of directors meeting minutes recording strategic decisions, budget approvals, policy amendments, and governance oversight activities",
        "Internal audit report examining operational procedures, compliance adherence, risk management effectiveness, and process improvement recommendations",
        "Employee disciplinary action documentation detailing misconduct investigation, due process adherence, corrective measures, and appeal procedures",
        "Workforce planning strategy addressing staffing requirements, skill gap analysis, succession planning, and talent retention initiatives",
        "Code of conduct manual outlining ethical standards, behavioral expectations, conflict resolution procedures, and disciplinary consequences",
        "Job description for a station master position, detailing responsibilities, qualifications, and reporting structure",
        "Employee handbook outlining company policies, benefits, leave application procedures, and code of conduct",
        "Internal memo from the HR department regarding upcoming training sessions for new customer service software",
        "Meeting agenda for a departmental head meeting, listing discussion topics, action items, and attendees",
        "Annual leave request form for an employee, requiring manager approval and a record of leave balance",
        "Travel expense report for an administrative staff member, with receipts for travel, accommodation, and meals",
        "Confidentiality agreement for new employees, outlining data protection policies and non-disclosure clauses",
        "Record of employee attendance and punctuality for a specific month, used for performance tracking",
        "Internal memorandum regarding a new policy on remote work and flexible working hours for administrative staff",
        "Exit interview form for a departing employee, collecting feedback on their experience and reasons for leaving",
        "Workforce diversity report, analyzing the representation of different demographics across all departments",
        "Internal communication plan for a major organizational change, outlining the messaging and timeline for all employees",
        "Recruitment sourcing plan detailing the channels for finding candidates for open positions, including job boards and social media",
        "Onboarding checklist for new hires, ensuring all necessary paperwork, IT access, and training are completed",
        "Employee wellness program policy, outlining the benefits and resources available for physical and mental health",
        "Minutes of a safety committee meeting, documenting discussions on workplace hazards, safety training, and incident reports",
        "Disciplinary review documentation for a staff member who violated company policy, including a summary of the investigation and the final decision",
        "Performance improvement plan for an employee, outlining specific goals, a timeline for improvement, and a follow-up schedule",
        "Internal memo on a new policy for expense reimbursement, detailing the required documentation and approval process",
        "Job application form from a candidate for a management position, including their resume and cover letter"
    ],
    'Safety & Regulatory Documents': [
        "Emergency response plan outlining evacuation procedures, communication protocols, coordination with external agencies, and crisis management strategies",
        "Security threat assessment report analyzing vulnerability points, risk mitigation strategies, surveillance system effectiveness, and access control measures",
        "Environmental impact study documenting noise pollution levels, air quality effects, ecological preservation measures, and mitigation strategies",
        "Fire safety compliance certificate verifying sprinkler systems, emergency lighting, smoke detection equipment, and evacuation route adequacy",
        "Accident investigation report analyzing incident causes, safety protocol failures, human factors, and preventive action recommendations",
        "Regulatory compliance audit documenting adherence to transportation authority guidelines, safety standards, and operational requirements",
        "CCTV surveillance system operational manual covering camera positioning, recording procedures, data retention, and privacy protection measures",
        "Hazardous material handling protocol for maintenance chemicals, emergency response procedures, worker safety protection, and environmental compliance",
        "Safety management system documentation covering risk assessment, incident reporting, corrective action tracking, and continuous improvement processes",
        "Occupational health and safety policy defining workplace safety standards, personal protective equipment requirements, and injury prevention measures",
        "Risk assessment report for a new construction site, identifying potential hazards and outlining mitigation strategies for worker safety",
        "Emergency drill report documenting the results of a fire evacuation drill at a station, including response times and areas for improvement",
        "Regulatory inspection report from the transportation authority, detailing findings on operational safety and compliance with federal guidelines",
        "Procedure manual for handling bomb threats, including communication protocols, evacuation plans, and coordination with law enforcement",
        "Security audit report on the metro's access control systems, evaluating the effectiveness of keycard and biometric scanners",
        "Environmental management plan for the disposal of waste materials from station cleaning and maintenance operations",
        "Public health advisory from the regulatory body regarding sanitation standards on public transport and recommended hygiene protocols",
        "Incident report for a minor collision between two trains, detailing the cause, timeline of events, and actions taken",
        "Cybersecurity policy for the metro's IT network, outlining security protocols for data protection and threat response",
        "Operational safety audit checklist for all stations, verifying the availability of fire extinguishers, first aid kits, and emergency lighting",
        "Regulatory filing document with the government agency for a new line extension, including environmental impact statements and public hearing transcripts",
        "Terrorism threat assessment report, identifying high-risk areas and recommending enhanced security patrols and surveillance",
        "Protocol for handling a chemical spill on a train or station platform, outlining the steps for containment, evacuation, and cleanup",
        "Emergency communication plan for a power outage, detailing the use of backup generators and manual signaling systems",
        "Report on the effectiveness of the security surveillance system, with statistics on crime rates and incident response times",
        "Worker safety handbook outlining procedures for working on live tracks and using heavy machinery safely",
        "Audit report on a station's emergency exits, confirming that all exits are clearly marked and fully functional",
        "Incident investigation summary for a passenger injury, detailing the circumstances and recommended preventive actions",
        "Regulatory update memo from the transportation authority, outlining new safety standards and compliance deadlines",
        "Security protocol for handling large crowds and special events at metro stations, including crowd control measures and emergency response plans"
    ],
    'Strategic & Project Management Documents': [
        "Five-year strategic development plan outlining network expansion objectives, ridership targets, revenue growth projections, and investment priorities",
        "Detailed project report for new metro line construction including feasibility analysis, cost estimates, implementation timeline, and stakeholder impact",
        "Feasibility study for automated metro system implementation covering technical requirements, financial viability, and operational benefits",
        "Project milestone review documenting construction progress, budget utilization, schedule adherence, risk management, and quality control measures",
        "Ridership demand forecasting model incorporating demographic trends, urban development patterns, and multimodal transportation integration",
        "Integration masterplan for multimodal connectivity linking metro, bus rapid transit, railway networks, and first-last mile connectivity",
        "Technology roadmap for smart metro initiatives including digital ticketing, real-time information systems, predictive maintenance, and IoT integration",
        "Stakeholder engagement strategy for community consultation, government coordination, public-private partnerships, and regulatory approval processes",
        "Business case development for metro extension project covering market analysis, financial projections, economic impact assessment, and funding options",
        "Change management framework for organizational transformation, technology adoption, process improvement, and cultural change initiatives",
        "Project charter for a new station design project, outlining the scope, objectives, key stakeholders, and project team members",
        "Risk management plan for a major infrastructure project, identifying potential risks and outlining mitigation and contingency strategies",
        "Gantt chart for the construction of a new depot, showing a timeline for each task and its dependencies",
        "Feasibility study for the expansion of the metro's advertising revenue, including market analysis and financial projections",
        "Strategic plan for improving operational efficiency, outlining key performance indicators and goals for the next three years",
        "Project status report for a signaling system upgrade, detailing progress, budget status, and upcoming milestones",
        "Stakeholder analysis matrix for a public-private partnership project, identifying key stakeholders and their interests",
        "Detailed business plan for a new metro line, including an executive summary, market analysis, and financial model",
        "Lessons learned document from a completed project, summarizing what went well and what could be improved for future projects",
        "Strategic review of the metro's customer service operations, outlining goals for improving passenger satisfaction and loyalty",
        "Project budget plan for a new technology implementation, detailing the allocation of funds for hardware, software, and personnel costs",
        "Business case for a new mobile application development, including a market analysis, financial projections, and risk assessment",
        "Change management plan for the implementation of a new employee management software, outlining communication and training strategies",
        "Project portfolio report summarizing the status of all active projects, including their progress, budget, and strategic alignment",
        "Strategic plan for improving energy efficiency across the metro system, with a timeline for implementation and expected savings",
        "Project management dashboard report, providing a visual overview of key project metrics like schedule, budget, and resource allocation",
        "Business case for a new smart card system, including a cost-benefit analysis and a plan for implementation",
        "Feasibility study for a new station location, including a demographic analysis of the area and potential ridership numbers",
        "Risk mitigation plan for a new train procurement project, addressing potential issues with supplier reliability and component quality",
        "Strategic report on the future of the metro network, outlining potential expansion opportunities and new service offerings"
    ]
}

        data = []
        for category, docs in sample_docs.items():
            for doc in docs:
                data.append({'text': doc, 'category': category})

        df = pd.DataFrame(data)
        logger.info(f"Created sample dataset with {len(df)} documents across {df['category'].nunique()} categories")
        return df

    def prepare_data(self, df, test_size=0.2, stratify=True):
        """Prepare data for BERT training"""
        if df is None or len(df) == 0:
            raise ValueError("No data provided for training")

        df['text'] = df['text'].astype(str).str.strip()
        initial_count = len(df)
        df = df[df['text'].str.len() > 10]
        logger.info(f"Removed {initial_count - len(df)} documents with insufficient text")

        df['label'] = df['category'].map(self.label_to_id)
        unknown_labels = df['label'].isna().sum()
        if unknown_labels > 0:
            logger.warning(f"Found {unknown_labels} documents with unknown categories")
            df = df.dropna(subset=['label'])

        stratify_labels = df['label'] if stratify else None
        train_texts, val_texts, train_labels, val_labels = train_test_split(
            df['text'].tolist(),
            df['label'].tolist(),
            test_size=test_size,
            random_state=42,
            stratify=stratify_labels
        )

        train_dataset = MetroDocumentDataset(
            train_texts, train_labels, self.tokenizer,
            self.max_length, self.label_to_id
        )
        val_dataset = MetroDocumentDataset(
            val_texts, val_labels, self.tokenizer,
            self.max_length, self.label_to_id
        )

        logger.info(f"Training set: {len(train_dataset)} documents")
        logger.info(f"Validation set: {len(val_dataset)} documents")

        return train_dataset, val_dataset, val_texts, val_labels

    def compute_metrics(self, eval_pred):
        """Enhanced evaluation metrics computation"""
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)

        accuracy = accuracy_score(labels, predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, predictions, average='macro', zero_division=0
        )

        return {
            'accuracy': accuracy,
            'f1': f1,
            'precision': precision,
            'recall': recall
        }

    def train(self, train_dataset, val_dataset, num_epochs=3, batch_size=4,
              learning_rate=2e-5, output_dir='/content/bert_results'):
        """Enhanced training with early stopping and logging"""

        Path(output_dir).mkdir(parents=True, exist_ok=True)

        self.model = AutoModelForSequenceClassification.from_pretrained(
            self.model_name,
            num_labels=self.num_labels,
            problem_type="single_label_classification"
        ).to(self.device) # Move model to device

        training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=num_epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            learning_rate=learning_rate,
            warmup_ratio=0.1,
            weight_decay=0.01,
            logging_dir=f'{output_dir}/logs',
            logging_steps=10,
            eval_strategy='epoch',
            save_strategy='epoch',
            load_best_model_at_end=True,
            metric_for_best_model='f1',
            greater_is_better=True,
            seed=42,
            fp16=torch.cuda.is_available(),
            report_to=None,
            save_total_limit=2,
        )

        self.trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            compute_metrics=self.compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
        )

        logger.info("Starting BERT training...")
        start_time = datetime.now()

        try:
            training_result = self.trainer.train()
            training_time = datetime.now() - start_time

            # Calculate epochs completed based on steps and dataset size
            steps_per_epoch = len(train_dataset) // batch_size
            epochs_completed = training_result.global_step / steps_per_epoch if steps_per_epoch > 0 else 0

            self.training_history = {
                'training_time': str(training_time),
                'train_loss': training_result.training_loss,
                'epochs_completed': epochs_completed,
                'global_step': training_result.global_step
            }

            eval_results = self.trainer.evaluate()
            self.training_history.update(eval_results)

            logger.info("Training completed successfully!")
            logger.info(f"Training time: {training_time}")
            logger.info(f"Final validation accuracy: {eval_results['eval_accuracy']:.3f}")
            logger.info(f"Final validation F1-score: {eval_results['eval_f1']:.3f}")

            return eval_results

        except Exception as e:
            logger.error(f"Training failed: {str(e)}")
            raise

    def predict(self, text, return_all_scores=False):
        """Enhanced prediction with confidence scores"""
        if self.model is None:
            raise ValueError("Model not trained yet!")

        inputs = self.tokenizer(
            text,
            truncation=True,
            padding=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        # Move inputs to the same device as the model
        inputs = {name: tensor.to(self.device) for name, tensor in inputs.items()}

        self.model.eval()
        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits
            probabilities = F.softmax(logits, dim=-1).squeeze()

        # Ensure probabilities tensor is also on the correct device before converting to numpy
        probs = probabilities.cpu().numpy()

        if return_all_scores:
            predictions = []
            for i, prob in enumerate(probs):
                category = self.id_to_label[i]
                predictions.append((category, float(prob)))
            predictions = sorted(predictions, key=lambda x: x[1], reverse=True)
            return predictions
        else:
            predicted_idx = np.argmax(probs)
            predicted_category = self.id_to_label[predicted_idx]
            confidence = float(probs[predicted_idx])
            return predicted_category, confidence

    def save_model(self, path):
        """Save trained model and tokenizer"""
        if self.model is None:
            raise ValueError("No model to save!")

        save_path = Path(path)
        save_path.mkdir(parents=True, exist_ok=True)

        self.model.save_pretrained(save_path)
        self.tokenizer.save_pretrained(save_path)

        metadata = {
            'categories': self.categories,
            'label_to_id': self.label_to_id,
            'id_to_label': self.id_to_label,
            'model_name': self.model_name,
            'max_length': self.max_length,
            'training_history': self.training_history
        }

        with open(save_path / 'metadata.json', 'w') as f:
            json.dump(metadata, f, indent=2)

        logger.info(f"Model saved to {save_path}")


    def load_model(self, path):
        """Load trained model and tokenizer"""
        from pathlib import Path
        import json

        load_path = Path(path)

        if not load_path.exists():
            raise FileNotFoundError(f"Model path {load_path} does not exist")

        self.model = AutoModelForSequenceClassification.from_pretrained(load_path).to(self.device) # Move model to device
        self.tokenizer = AutoTokenizer.from_pretrained(load_path)

        metadata_path = load_path / 'metadata.json'
        if metadata_path.exists():
            with open(metadata_path, 'r') as f:
                metadata = json.load(f)

            self.categories = metadata.get('categories', self.categories)
            self.label_to_id = metadata.get('label_to_id', self.label_to_id)
            self.id_to_label = metadata.get('id_to_label', self.id_to_label)
            self.training_history = metadata.get('training_history', {})

        logger.info(f"Model loaded from {load_path}")


print("✅ Dataset and Enhanced BERT Classifier classes defined successfully!")

✅ Dataset and Enhanced BERT Classifier classes defined successfully!


In [ ]:
class EnhancedBERTDocumentClassifier:
    """Enhanced BERT classifier for metro railway documents"""

    def __init__(self, model_name='bert-base-uncased', num_labels=6, max_length=512):
        self.model_name = model_name
        self.num_labels = num_labels
        self.max_length = max_length
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = None
        self.trainer = None
        self.training_history = {}
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


        self.categories = [
            'Technical & Engineering Documents',
            'Passenger & Public-Facing Documents',
            'Financial & Procurement Documents',
            'Human Resources & Administrative Documents',
            'Safety, Security & Regulatory Documents',
            'Strategic & Project Management Documents'
        ]

        self.label_to_id = {label: idx for idx, label in enumerate(self.categories)}
        self.id_to_label = {idx: label for idx, label in enumerate(self.categories)}

        logger.info(f"Initialized classifier with {self.num_labels} categories")

    def prepare_enhanced_sample_data(self):
        """Create enhanced sample training data"""
        sample_docs = {
    'Technical & Engineering Documents': [
        "Architectural blueprints and structural engineering designs for metro station construction with load-bearing calculations and seismic analysis requirements",
        "Electrical schematic diagrams for 25kV overhead catenary system power distribution network installation with substation specifications",
        "Track geometry inspection report documenting rail alignment deviations, gauge measurements, and corrective maintenance requirements",
        "Automated train control signaling system specifications including CBTC communication protocols, ATP safety interlocks, and ATO functionality",
        "Rolling stock technical manual covering traction motors, regenerative braking systems, HVAC equipment maintenance, and door control mechanisms",
        "Civil engineering soil analysis report for tunnel boring machine operation, ground settlement monitoring, and foundation stability assessment",
        "Mechanical ventilation system design specifications for underground station air circulation, smoke extraction, and emergency exhaust capabilities",
        "Platform screen door installation guidelines with electrical interfacing, passenger safety protocols, and emergency override procedures",
        "Signal maintenance procedure documenting LED aspect replacement, cable testing, and communication system troubleshooting methods",
        "Power supply system technical specifications including transformer ratings, protective relay settings, and fault detection mechanisms",
        "Design review documentation for the new elevated track section, focusing on material stress tolerance and expansion joint placement",
        "Geotechnical survey results for a tunnel expansion project, including rock stability analysis and groundwater management plans",
        "Maintenance log for automated fare collection gates, detailing software updates, hardware repairs, and calibration records",
        "System integration protocol for connecting a new security surveillance network with the existing IT infrastructure and data center",
        "Technical drawing for a traction substation layout, including the positioning of rectifiers, transformers, and switchgear for high-voltage power conversion",
        "Failure analysis report for a train's auxiliary power supply unit, identifying the root cause of the component failure and recommending a design change",
        "Installation manual for a fiber optic communication network, specifying cable routing, splicing procedures, and signal testing protocols",
        "Detailed plan for the construction of an underground pedestrian walkway, including structural supports, lighting design, and drainage systems",
        "Blueprint for the extension of the metro depot, showing the layout of new maintenance tracks, inspection pits, and workshop facilities",
        "Specifications for new signaling equipment, including a list of approved vendors and compliance requirements for train control system integration",
        "Software update procedure for the central dispatch system, detailing the rollback plan, testing protocols, and version control",
        "Report on the structural integrity of a viaduct, based on a recent drone inspection and material fatigue analysis",
        "Specifications for specialized welding equipment used for track repair, including safety features and operational procedures",
        "Hydraulic system maintenance manual for train braking systems, detailing fluid replacement schedules and pressure testing procedures",
        "Electrical power consumption report for an entire metro line, analyzing energy usage by station and train operation patterns",
        "Detailed schematics for the station's fire suppression system, including sprinkler head placement and control panel wiring diagrams",
        "Technical proposal for a smart lighting system to be installed in all metro tunnels, including energy-saving features and remote monitoring capabilities",
        "Report on rail wear and tear analysis, recommending a schedule for rail grinding and replacement based on operational data",
        "Design document for the installation of new communication towers along the metro line, including tower specifications and radio frequency planning",
        "Project plan for the upgrading of all station escalators to new, more energy-efficient models with safety sensor integrations"
    ],
    'Passenger & Public-Facing Documents': [
        "Metro system route map displaying all lines, stations, interchange connections, and accessibility features for passenger navigation assistance",
        "Customer service complaint form documenting passenger feedback regarding train delays, overcrowding, and service quality concerns",
        "Fare structure notification announcing revised tariff rates, distance-based pricing, concession categories, and smart card benefits",
        "Accessibility guide for disabled passengers outlining elevator locations, tactile guidance systems, and staff assistance procedures",
        "Public announcement script for service interruptions, alternative transportation options, estimated restoration times, and passenger safety instructions",
        "Smart card user manual explaining top-up procedures, balance checking methods, automatic fare collection system, and card replacement process",
        "Station facility information including restroom locations, commercial areas, parking availability, bicycle storage, and connecting bus services",
        "Emergency passenger communication protocol for evacuation procedures, safety instructions during incidents, and coordination with emergency services",
        "Mobile application user guide covering real-time train information, journey planning, fare calculation, and customer feedback submission",
        "Passenger satisfaction survey questionnaire collecting feedback on service quality, cleanliness, punctuality, and overall travel experience",
        "Press release announcing the opening of a new metro line and its key features for public awareness and media engagement",
        "Public safety poster detailing emergency contact numbers, fire extinguisher locations, and safe boarding procedures at the station",
        "Lost and found claim form for passengers to report missing items and provide contact information for retrieval",
        "User guide for the new ticketing kiosks, explaining how to select a destination, pay with a credit card, and collect a ticket",
        "Community engagement report for the new metro line project, summarizing public feedback and addressing local concerns",
        "Passenger rights charter outlining the standards of service, compensation policies for delays, and complaint resolution procedures",
        "Guide to local attractions and public transport links available from each metro station, with opening hours and ticket information",
        "Promotional leaflet for a new monthly travel pass, highlighting the cost savings and convenience for daily commuters",
        "Code of conduct for passengers, outlining rules on eating, drinking, and appropriate behavior within metro premises",
        "Annual report on customer satisfaction metrics, including ratings for staff helpfulness, station cleanliness, and ticket accessibility",
        "Detailed station evacuation map showing all exits, emergency exits, and assembly points for public reference",
        "Instructions for how to use the station's public WiFi network, including login procedures and data usage policies",
        "List of prohibited items on the metro, with illustrations and safety warnings for passengers",
        "Public notice about construction work near a metro station, advising passengers of temporary closures and alternative routes",
        "Feedback form for accessibility features, allowing disabled passengers to provide specific comments on elevator and ramp availability",
        "Service disruption alert message for a line closure, providing a link to a temporary bus shuttle schedule",
        "Map showing all designated smoking areas and waste disposal locations within the metro system",
        "Passenger guide to using the escalators and elevators safely, with specific warnings for children and luggage",
        "Promotional brochure for a new metro line extension, showcasing a virtual tour of the future stations and key landmarks",
        "Public bulletin regarding changes to operating hours during a public holiday, advising passengers on the last train times"
    ],
    'Tenders & Procurement': [
        "Annual financial statement with comprehensive revenue analysis, operational expenditure breakdown, capital investment summary, and profitability metrics",
        "Request for proposal document for metro car procurement specifying technical requirements, delivery schedules, evaluation criteria, and contract terms",
        "Contractor payment voucher for civil works completion with milestone verification, quality compliance certification, and performance bond details",
        "Vendor service level agreement defining performance metrics, availability targets, penalty clauses, maintenance responsibility scope, and escalation procedures",
        "Budget allocation report for infrastructure development, rolling stock acquisition, operational cost projections, and contingency fund management",
        "Audit findings report highlighting financial irregularities, compliance gaps, internal control weaknesses, and recommended corrective actions",
        "Tender evaluation committee report comparing bidder proposals on technical merit, commercial viability, past performance, and financial capability",
        "Procurement policy document establishing vendor qualification criteria, bidding procedures, contract award process, and conflict of interest guidelines",
        "Cost-benefit analysis for metro line extension project including construction costs, ridership projections, revenue forecasts, and payback period",
        "Invoice processing workflow documentation covering approval hierarchy, payment terms, tax calculations, and dispute resolution procedures",
        "Quarterly financial report with detailed balance sheet, income statement, and cash flow analysis for investor presentation",
        "Purchase order for signal equipment, detailing the quantity, unit price, delivery date, and total cost of the components",
        "Memorandum of Understanding with a private firm for the co-development of a commercial complex at a metro station site",
        "Request for quotation for station furniture, including specifications for benches, waste bins, and information kiosks",
        "Treasury report summarizing daily cash balances, investment portfolio performance, and short-term liquidity projections",
        "Disbursement request for the payment of a contractor's final invoice, with a checklist for verification and approval",
        "Grant proposal application for government funding of a new metro line, outlining the project's financial needs and economic benefits",
        "Annual budget plan for the maintenance and repair department, with a breakdown of labor costs, spare parts inventory, and external service contracts",
        "Contract amendment for a construction project, detailing changes to the project scope, timeline, and associated costs",
        "Due diligence report on a potential acquisition of a land parcel for a new station, including property valuation and legal title review",
        "Statement of work for a software development project for the metro's ticketing system, including deliverables, milestones, and payment schedule",
        "Procurement strategy document for a major project, outlining the sourcing plan, risk mitigation, and supplier management policies",
        "Cost estimation spreadsheet for the construction of a new tunnel section, including material costs, labor rates, and contingency allowances",
        "Bid submission form from a vendor for a rolling stock maintenance contract, detailing their technical and commercial offer",
        "Financial projections for a new metro line, including estimated revenue from fares, advertising, and retail spaces",
        "Internal audit report on the procurement process, highlighting areas of non-compliance and making recommendations for process improvement",
        "Budgetary approval memo for the purchase of new security cameras for all stations, outlining the justification and fund source",
        "Investment analysis report for a new technology adoption, detailing the return on investment and potential cost savings over a five-year period",
        "Credit rating report for the metro company, assessing its financial stability and ability to meet its debt obligations",
        "Invoice from a supplier for electrical components, listing the quantity, price per unit, and total amount due"
    ],
    'Human Resources & Administrative Documents': [
        "Employee recruitment policy defining selection criteria, interview process, background verification, probationary period requirements, and onboarding procedures",
        "Staff training program curriculum for technical skills development, safety procedures, customer service excellence, and professional certification requirements",
        "Performance evaluation report assessing employee productivity, goal achievement, competency development, career progression, and training needs",
        "Organizational restructuring memo announcing departmental changes, reporting relationships, role redefinitions, and transition timelines",
        "Salary revision notification implementing pay scale adjustments, allowance modifications, incentive structure changes, and benefit enhancements",
        "Board of directors meeting minutes recording strategic decisions, budget approvals, policy amendments, and governance oversight activities",
        "Internal audit report examining operational procedures, compliance adherence, risk management effectiveness, and process improvement recommendations",
        "Employee disciplinary action documentation detailing misconduct investigation, due process adherence, corrective measures, and appeal procedures",
        "Workforce planning strategy addressing staffing requirements, skill gap analysis, succession planning, and talent retention initiatives",
        "Code of conduct manual outlining ethical standards, behavioral expectations, conflict resolution procedures, and disciplinary consequences",
        "Job description for a station master position, detailing responsibilities, qualifications, and reporting structure",
        "Employee handbook outlining company policies, benefits, leave application procedures, and code of conduct",
        "Internal memo from the HR department regarding upcoming training sessions for new customer service software",
        "Meeting agenda for a departmental head meeting, listing discussion topics, action items, and attendees",
        "Annual leave request form for an employee, requiring manager approval and a record of leave balance",
        "Travel expense report for an administrative staff member, with receipts for travel, accommodation, and meals",
        "Confidentiality agreement for new employees, outlining data protection policies and non-disclosure clauses",
        "Record of employee attendance and punctuality for a specific month, used for performance tracking",
        "Internal memorandum regarding a new policy on remote work and flexible working hours for administrative staff",
        "Exit interview form for a departing employee, collecting feedback on their experience and reasons for leaving",
        "Workforce diversity report, analyzing the representation of different demographics across all departments",
        "Internal communication plan for a major organizational change, outlining the messaging and timeline for all employees",
        "Recruitment sourcing plan detailing the channels for finding candidates for open positions, including job boards and social media",
        "Onboarding checklist for new hires, ensuring all necessary paperwork, IT access, and training are completed",
        "Employee wellness program policy, outlining the benefits and resources available for physical and mental health",
        "Minutes of a safety committee meeting, documenting discussions on workplace hazards, safety training, and incident reports",
        "Disciplinary review documentation for a staff member who violated company policy, including a summary of the investigation and the final decision",
        "Performance improvement plan for an employee, outlining specific goals, a timeline for improvement, and a follow-up schedule",
        "Internal memo on a new policy for expense reimbursement, detailing the required documentation and approval process",
        "Job application form from a candidate for a management position, including their resume and cover letter"
    ],
    'Safety & Regulatory Documents': [
        "Emergency response plan outlining evacuation procedures, communication protocols, coordination with external agencies, and crisis management strategies",
        "Security threat assessment report analyzing vulnerability points, risk mitigation strategies, surveillance system effectiveness, and access control measures",
        "Environmental impact study documenting noise pollution levels, air quality effects, ecological preservation measures, and mitigation strategies",
        "Fire safety compliance certificate verifying sprinkler systems, emergency lighting, smoke detection equipment, and evacuation route adequacy",
        "Accident investigation report analyzing incident causes, safety protocol failures, human factors, and preventive action recommendations",
        "Regulatory compliance audit documenting adherence to transportation authority guidelines, safety standards, and operational requirements",
        "CCTV surveillance system operational manual covering camera positioning, recording procedures, data retention, and privacy protection measures",
        "Hazardous material handling protocol for maintenance chemicals, emergency response procedures, worker safety protection, and environmental compliance",
        "Safety management system documentation covering risk assessment, incident reporting, corrective action tracking, and continuous improvement processes",
        "Occupational health and safety policy defining workplace safety standards, personal protective equipment requirements, and injury prevention measures",
        "Risk assessment report for a new construction site, identifying potential hazards and outlining mitigation strategies for worker safety",
        "Emergency drill report documenting the results of a fire evacuation drill at a station, including response times and areas for improvement",
        "Regulatory inspection report from the transportation authority, detailing findings on operational safety and compliance with federal guidelines",
        "Procedure manual for handling bomb threats, including communication protocols, evacuation plans, and coordination with law enforcement",
        "Security audit report on the metro's access control systems, evaluating the effectiveness of keycard and biometric scanners",
        "Environmental management plan for the disposal of waste materials from station cleaning and maintenance operations",
        "Public health advisory from the regulatory body regarding sanitation standards on public transport and recommended hygiene protocols",
        "Incident report for a minor collision between two trains, detailing the cause, timeline of events, and actions taken",
        "Cybersecurity policy for the metro's IT network, outlining security protocols for data protection and threat response",
        "Operational safety audit checklist for all stations, verifying the availability of fire extinguishers, first aid kits, and emergency lighting",
        "Regulatory filing document with the government agency for a new line extension, including environmental impact statements and public hearing transcripts",
        "Terrorism threat assessment report, identifying high-risk areas and recommending enhanced security patrols and surveillance",
        "Protocol for handling a chemical spill on a train or station platform, outlining the steps for containment, evacuation, and cleanup",
        "Emergency communication plan for a power outage, detailing the use of backup generators and manual signaling systems",
        "Report on the effectiveness of the security surveillance system, with statistics on crime rates and incident response times",
        "Worker safety handbook outlining procedures for working on live tracks and using heavy machinery safely",
        "Audit report on a station's emergency exits, confirming that all exits are clearly marked and fully functional",
        "Incident investigation summary for a passenger injury, detailing the circumstances and recommended preventive actions",
        "Regulatory update memo from the transportation authority, outlining new safety standards and compliance deadlines",
        "Security protocol for handling large crowds and special events at metro stations, including crowd control measures and emergency response plans"
    ],
    'Strategic & Project Management Documents': [
        "Five-year strategic development plan outlining network expansion objectives, ridership targets, revenue growth projections, and investment priorities",
        "Detailed project report for new metro line construction including feasibility analysis, cost estimates, implementation timeline, and stakeholder impact",
        "Feasibility study for automated metro system implementation covering technical requirements, financial viability, and operational benefits",
        "Project milestone review documenting construction progress, budget utilization, schedule adherence, risk management, and quality control measures",
        "Ridership demand forecasting model incorporating demographic trends, urban development patterns, and multimodal transportation integration",
        "Integration masterplan for multimodal connectivity linking metro, bus rapid transit, railway networks, and first-last mile connectivity",
        "Technology roadmap for smart metro initiatives including digital ticketing, real-time information systems, predictive maintenance, and IoT integration",
        "Stakeholder engagement strategy for community consultation, government coordination, public-private partnerships, and regulatory approval processes",
        "Business case development for metro extension project covering market analysis, financial projections, economic impact assessment, and funding options",
        "Change management framework for organizational transformation, technology adoption, process improvement, and cultural change initiatives",
        "Project charter for a new station design project, outlining the scope, objectives, key stakeholders, and project team members",
        "Risk management plan for a major infrastructure project, identifying potential risks and outlining mitigation and contingency strategies",
        "Gantt chart for the construction of a new depot, showing a timeline for each task and its dependencies",
        "Feasibility study for the expansion of the metro's advertising revenue, including market analysis and financial projections",
        "Strategic plan for improving operational efficiency, outlining key performance indicators and goals for the next three years",
        "Project status report for a signaling system upgrade, detailing progress, budget status, and upcoming milestones",
        "Stakeholder analysis matrix for a public-private partnership project, identifying key stakeholders and their interests",
        "Detailed business plan for a new metro line, including an executive summary, market analysis, and financial model",
        "Lessons learned document from a completed project, summarizing what went well and what could be improved for future projects",
        "Strategic review of the metro's customer service operations, outlining goals for improving passenger satisfaction and loyalty",
        "Project budget plan for a new technology implementation, detailing the allocation of funds for hardware, software, and personnel costs",
        "Business case for a new mobile application development, including a market analysis, financial projections, and risk assessment",
        "Change management plan for the implementation of a new employee management software, outlining communication and training strategies",
        "Project portfolio report summarizing the status of all active projects, including their progress, budget, and strategic alignment",
        "Strategic plan for improving energy efficiency across the metro system, with a timeline for implementation and expected savings",
        "Project management dashboard report, providing a visual overview of key project metrics like schedule, budget, and resource allocation",
        "Business case for a new smart card system, including a cost-benefit analysis and a plan for implementation",
        "Feasibility study for a new station location, including a demographic analysis of the area and potential ridership numbers",
        "Risk mitigation plan for a new train procurement project, addressing potential issues with supplier reliability and component quality",
        "Strategic report on the future of the metro network, outlining potential expansion opportunities and new service offerings"
    ]
}

        data = []
        for category, docs in sample_docs.items():
            for doc in docs:
                data.append({'text': doc, 'category': category})

        df = pd.DataFrame(data)
        logger.info(f"Created sample dataset with {len(df)} documents across {df['category'].nunique()} categories")
        return df

print("✅ Enhanced BERT Classifier class defined successfully!")

✅ Enhanced BERT Classifier class defined successfully!


In [ ]:
# Continue with classifier methods
def prepare_data(self, df, test_size=0.2, stratify=True):
    """Prepare data for BERT training"""
    if df is None or len(df) == 0:
        raise ValueError("No data provided for training")

    df['text'] = df['text'].astype(str).str.strip()
    initial_count = len(df)
    df = df[df['text'].str.len() > 10]
    logger.info(f"Removed {initial_count - len(df)} documents with insufficient text")

    df['label'] = df['category'].map(self.label_to_id)
    unknown_labels = df['label'].isna().sum()
    if unknown_labels > 0:
        logger.warning(f"Found {unknown_labels} documents with unknown categories")
        df = df.dropna(subset=['label'])

    stratify_labels = df['label'] if stratify else None
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        df['text'].tolist(),
        df['label'].tolist(),
        test_size=test_size,
        random_state=42,
        stratify=stratify_labels
    )

    train_dataset = MetroDocumentDataset(
        train_texts, train_labels, self.tokenizer,
        self.max_length, self.label_to_id
    )
    val_dataset = MetroDocumentDataset(
        val_texts, val_labels, self.tokenizer,
        self.max_length, self.label_to_id
    )

    logger.info(f"Training set: {len(train_dataset)} documents")
    logger.info(f"Validation set: {len(val_dataset)} documents")

    return train_dataset, val_dataset, val_texts, val_labels

def compute_metrics(self, eval_pred):
    """Enhanced evaluation metrics computation"""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='macro', zero_division=0
    )

    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Add methods to classifier class
EnhancedBERTDocumentClassifier.prepare_data = prepare_data
EnhancedBERTDocumentClassifier.compute_metrics = compute_metrics

print("✅ Classifier data preparation methods added!")

✅ Classifier data preparation methods added!


In [ ]:
# Training and prediction methods
def train(self, train_dataset, val_dataset, num_epochs=3, batch_size=4,
          learning_rate=2e-5, output_dir='/content/bert_results'):
    """Enhanced training with early stopping and logging"""

    Path(output_dir).mkdir(parents=True, exist_ok=True)

    self.model = AutoModelForSequenceClassification.from_pretrained(
        self.model_name,
        num_labels=self.num_labels,
        problem_type="single_label_classification"
    ).to(self.device) # Move model to device

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=learning_rate,
        warmup_ratio=0.1,
        weight_decay=0.01,
        logging_dir=f'{output_dir}/logs',
        logging_steps=10,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        greater_is_better=True,
        seed=42,
        fp16=torch.cuda.is_available(),
        report_to=None,
        save_total_limit=2,
    )

    self.trainer = Trainer(
        model=self.model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=self.compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )

    logger.info("Starting BERT training...")
    start_time = datetime.now()

    try:
        training_result = self.trainer.train()
        training_time = datetime.now() - start_time

        # Calculate epochs completed based on steps and dataset size
        steps_per_epoch = len(train_dataset) // batch_size
        epochs_completed = training_result.global_step / steps_per_epoch if steps_per_epoch > 0 else 0

        self.training_history = {
            'training_time': str(training_time),
            'train_loss': training_result.training_loss,
            'epochs_completed': epochs_completed,
            'global_step': training_result.global_step
        }

        eval_results = self.trainer.evaluate()
        self.training_history.update(eval_results)

        logger.info("Training completed successfully!")
        logger.info(f"Training time: {training_time}")
        logger.info(f"Final validation accuracy: {eval_results['eval_accuracy']:.3f}")
        logger.info(f"Final validation F1-score: {eval_results['eval_f1']:.3f}")

        return eval_results

    except Exception as e:
        logger.error(f"Training failed: {str(e)}")
        raise

def predict(self, text, return_all_scores=False):
    """Enhanced prediction with confidence scores"""
    if self.model is None:
        raise ValueError("Model not trained yet!")

    inputs = self.tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=self.max_length,
        return_tensors='pt'
    )

    # Move inputs to the same device as the model
    inputs = {name: tensor.to(self.device) for name, tensor in inputs.items()}

    self.model.eval()
    with torch.no_grad():
        outputs = self.model(**inputs)
        logits = outputs.logits
        probabilities = F.softmax(logits, dim=-1).squeeze()

        # Ensure probabilities tensor is also on the correct device before converting to numpy
        probs = probabilities.cpu().numpy()


    if return_all_scores:
        predictions = []
        for i, prob in enumerate(probs):
            category = self.id_to_label[i]
            predictions.append((category, float(prob)))
        predictions = sorted(predictions, key=lambda x: x[1], reverse=True)
        return predictions
    else:
        predicted_idx = np.argmax(probs)
        predicted_category = self.id_to_label[predicted_idx]
        confidence = float(probs[predicted_idx])
        return predicted_category, confidence

def save_model(self, path):
    """Save trained model and tokenizer"""
    if self.model is None:
        raise ValueError("No model to save!")

    save_path = Path(path)
    save_path.mkdir(parents=True, exist_ok=True)

    self.model.save_pretrained(save_path)
    self.tokenizer.save_pretrained(save_path)

    metadata = {
        'categories': self.categories,
        'label_to_id': self.label_to_id,
        'id_to_label': self.id_to_label,
        'model_name': self.model_name,
        'max_length': self.max_length,
        'training_history': self.training_history
    }

    with open(save_path / 'metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2)

    logger.info(f"Model saved to {save_path}")

# Add methods to classifier class
EnhancedBERTDocumentClassifier.train = train
EnhancedBERTDocumentClassifier.predict = predict
EnhancedBERTDocumentClassifier.save_model = save_model

## 🎯 Step 3: Initialize and Prepare Data

Let's create the classifier and prepare our training data.

In [ ]:
# Initialize the enhanced BERT classifier
print("🚇 Initializing Enhanced BERT Document Classifier for Metro Railway")
print("=" * 70)

classifier = EnhancedBERTDocumentClassifier()

print(f"✅ Classifier initialized with {len(classifier.categories)} categories:")
for i, category in enumerate(classifier.categories, 1):
    print(f"  {i}. {category}")

print(f"\n📋 Model: {classifier.model_name}")
print(f"📋 Max text length: {classifier.max_length} tokens")

🚇 Initializing Enhanced BERT Document Classifier for Metro Railway
✅ Classifier initialized with 6 categories:
  1. Technical & Engineering Documents
  2. Passenger & Public-Facing Documents
  3. Financial & Procurement Documents
  4. Human Resources & Administrative Documents
  5. Safety, Security & Regulatory Documents
  6. Strategic & Project Management Documents

📋 Model: bert-base-uncased
📋 Max text length: 512 tokens


In [ ]:
# Create enhanced sample training data
print("📄 Preparing enhanced sample training data...")

df = classifier.prepare_enhanced_sample_data()

print(f"✅ Dataset created: {len(df)} documents")
print(f"✅ Categories: {df['category'].nunique()}")

# Show category distribution
print("\n📊 Category distribution:")
category_counts = df['category'].value_counts()
for category, count in category_counts.items():
    print(f"  • {category}: {count} documents")

# Show sample documents
print("\n📝 Sample documents:")
for i, (idx, row) in enumerate(df.head(3).iterrows()):
    print(f"\n{i+1}. Category: {row['category']}")
    print(f"   Text: {row['text'][:100]}...")

📄 Preparing enhanced sample training data...
✅ Dataset created: 180 documents
✅ Categories: 6

📊 Category distribution:
  • Technical & Engineering Documents: 30 documents
  • Passenger & Public-Facing Documents: 30 documents
  • Tenders & Procurement: 30 documents
  • Human Resources & Administrative Documents: 30 documents
  • Safety & Regulatory Documents: 30 documents
  • Strategic & Project Management Documents: 30 documents

📝 Sample documents:

1. Category: Technical & Engineering Documents
   Text: Architectural blueprints and structural engineering designs for metro station construction with load...

2. Category: Technical & Engineering Documents
   Text: Electrical schematic diagrams for 25kV overhead catenary system power distribution network installat...

3. Category: Technical & Engineering Documents
   Text: Track geometry inspection report documenting rail alignment deviations, gauge measurements, and corr...


In [ ]:
# Prepare data for training
print("🔄 Preparing data for BERT training...")

try:
    train_dataset, val_dataset, val_texts, val_labels = classifier.prepare_data(df)

    print(f"✅ Training dataset: {len(train_dataset)} samples")
    print(f"✅ Validation dataset: {len(val_dataset)} samples")

    # Show training data distribution
    import collections
    train_labels_str = [classifier.id_to_label[train_dataset[i]['labels'].item()]
                       for i in range(len(train_dataset))]
    label_counts = collections.Counter(train_labels_str)

    print("\n📊 Training data distribution:")
    for label, count in label_counts.items():
        print(f"  • {label}: {count} samples")

    print("\n✅ Data preparation completed successfully!")

except Exception as e:
    print(f"❌ Error preparing data: {e}")
    raise

🔄 Preparing data for BERT training...
✅ Training dataset: 96 samples
✅ Validation dataset: 24 samples

📊 Training data distribution:
  • Human Resources & Administrative Documents: 24 samples
  • Technical & Engineering Documents: 24 samples
  • Strategic & Project Management Documents: 24 samples
  • Passenger & Public-Facing Documents: 24 samples

✅ Data preparation completed successfully!


## 🚀 Step 4: Train the BERT Model

Now let's train our BERT model. This will take 15-45 minutes depending on your GPU.

In [ ]:
# Configure training parameters for Colab
print("⚙️ Configuring training parameters for Google Colab...")

# Check GPU memory and set appropriate batch size
if torch.cuda.is_available():
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Memory: {gpu_memory:.1f} GB")

    if gpu_memory < 10:
        batch_size = 2
        print(f"Using batch_size={batch_size} for limited GPU memory")
    elif gpu_memory < 15:
        batch_size = 4
        print(f"Using batch_size={batch_size} for moderate GPU memory")
    else:
        batch_size = 8
        print(f"Using batch_size={batch_size} for high GPU memory")
else:
    batch_size = 1
    print("Using CPU training with batch_size=1 (will be slower)")

# Training configuration
num_epochs = 10  # Increased for potentially better performance
learning_rate = 2e-5
output_dir = '/content/colab_bert_results'

print(f"\n📋 Training Configuration:")
print(f"  • Epochs: {num_epochs}")
print(f"  • Batch size: {batch_size}")
print(f"  • Learning rate: {learning_rate}")
print(f"  • Output directory: {output_dir}")
print(f"  • Estimated time: 15-30 minutes")

⚙️ Configuring training parameters for Google Colab...
GPU Memory: 15.8 GB
Using batch_size=8 for high GPU memory

📋 Training Configuration:
  • Epochs: 10
  • Batch size: 8
  • Learning rate: 2e-05
  • Output directory: /content/colab_bert_results
  • Estimated time: 15-30 minutes


In [ ]:
# Start BERT training
print("🎯 Starting BERT training...")
print("⏱️  This will take 15-45 minutes - please be patient!")
print("\n" + "="*50)

try:
    # Clear any cached GPU memory
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Start training
    eval_results = classifier.train(
        train_dataset=train_dataset,
        val_dataset=val_dataset,
        num_epochs=num_epochs,
        batch_size=batch_size,
        learning_rate=learning_rate,
        output_dir=output_dir
    )

    print("\n" + "="*50)
    print("🎉 TRAINING COMPLETED SUCCESSFULLY!")
    print("="*50)
    print(f"📊 Final Results:")
    print(f"  • Validation Accuracy: {eval_results['eval_accuracy']:.3f} ({eval_results['eval_accuracy']*100:.1f}%)")
    print(f"  • Validation F1-Score: {eval_results['eval_f1']:.3f}")
    print(f"  • Validation Precision: {eval_results['eval_precision']:.3f}")
    print(f"  • Validation Recall: {eval_results['eval_recall']:.3f}")

    if 'training_time' in classifier.training_history:
        print(f"  • Training Time: {classifier.training_history['training_time']}")

    training_success = True

except Exception as e:
    print(f"\n❌ Training failed: {e}")
    print("\n🔧 Troubleshooting tips:")
    print("  1. Try reducing batch_size to 1")
    print("  2. Restart runtime: Runtime → Restart runtime")
    print("  3. Clear GPU cache: torch.cuda.empty_cache()")
    print("  4. Switch to CPU training if GPU issues persist")

    training_success = False
    raise

🎯 Starting BERT training...
⏱️  This will take 15-45 minutes - please be patient!



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.795000,1.667806,0.250000,0.100000,0.062500,0.250000
2,1.586200,1.379171,0.625000,0.532418,0.465774,0.625000
3,1.352700,1.240377,0.666667,0.668137,0.761364,0.666667
4,1.112000,0.965881,0.791667,0.786713,0.792857,0.791667
5,0.740400,0.846069,0.750000,0.752697,0.784821,0.750000
6,0.565900,0.722239,0.750000,0.757576,0.788889,0.750000
7,0.453600,0.657257,0.750000,0.757576,0.788889,0.750000



🎉 TRAINING COMPLETED SUCCESSFULLY!
📊 Final Results:
  • Validation Accuracy: 0.792 (79.2%)
  • Validation F1-Score: 0.787
  • Validation Precision: 0.793
  • Validation Recall: 0.792
  • Training Time: 0:02:14.127986


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 🧪 Step 5: Test the Trained Model

Let's test our trained model on sample documents to see how well it performs.

In [ ]:
# Test the trained model on sample documents
if training_success:
    print("🧪 Testing the trained BERT model...")
    print("="*60)

    test_documents = [
        {
            'text': "Technical blueprint specifications for metro station platform design with structural engineering calculations and seismic resistance analysis for earthquake safety compliance",
            'expected': "Technical & Engineering Documents"
        },
        {
            'text': "Customer service feedback form documenting passenger complaints about train delays, overcrowding issues, and service quality concerns during peak operating hours",
            'expected': "Passenger & Public-Facing Documents"
        },
        {
            'text': "Annual financial audit report with comprehensive budget variance analysis, procurement expenditure review, and cost optimization recommendations for operational efficiency",
            'expected': "Financial & Procurement Documents"
        },
        {
            'text': "Employee performance evaluation procedures and staff training curriculum for professional development, safety compliance, and customer service excellence programs",
            'expected': "Human Resources & Administrative Documents"
        },
        {
            'text': "Emergency evacuation protocol with comprehensive fire safety procedures, regulatory compliance requirements, and passenger protection measures during crisis situations",
            'expected': "Safety, Security & Regulatory Documents"
        },
        {
            'text': "Five-year strategic expansion plan with network development priorities, ridership forecasting analysis, and investment allocation strategy for sustainable growth",
            'expected': "Strategic & Project Management Documents"
        }
    ]

    correct_predictions = 0
    total_predictions = len(test_documents)

    print(f"Testing {total_predictions} sample documents...\n")

    for i, doc in enumerate(test_documents, 1):
        try:
            # Get all predictions with scores
            predictions = classifier.predict(doc['text'], return_all_scores=True)
            predicted_category = predictions[0][0]
            confidence = predictions[0][1]

            # Check if prediction is correct
            is_correct = predicted_category == doc['expected']
            if is_correct:
                correct_predictions += 1

            status = "✅ CORRECT" if is_correct else "❌ INCORRECT"

            print(f"Document {i}: {status}")
            print(f"Text: {doc['text'][:80]}...")
            print(f"Expected: {doc['expected']}")
            print(f"Predicted: {predicted_category}")
            print(f"Confidence: {confidence:.3f}")

            # Show top 3 predictions
            print("Top 3 predictions:")
            for j, (category, conf) in enumerate(predictions[:3], 1):
                marker = "→" if j == 1 else " "
                print(f"  {marker} {j}. {category}: {conf:.3f}")
            print("-" * 60)

        except Exception as e:
            print(f"❌ Error predicting document {i}: {e}")
            print("-" * 60)

    # Calculate and display overall accuracy
    test_accuracy = correct_predictions / total_predictions
    print(f"\n📊 TEST RESULTS SUMMARY:")
    print(f"Correct predictions: {correct_predictions}/{total_predictions}")
    print(f"Test accuracy: {test_accuracy:.1%}")

    if test_accuracy >= 0.8:
        print("🎉 Excellent performance! Model is working well.")
    elif test_accuracy >= 0.6:
        print("👍 Good performance! Model shows promise.")
    else:
        print("⚠️  Performance could be improved with more training data.")

else:
    print("⚠️  Cannot test model - training was not successful.")
    print("Please fix training issues first.")

🧪 Testing the trained BERT model...
Testing 6 sample documents...

Document 1: ✅ CORRECT
Text: Technical blueprint specifications for metro station platform design with struct...
Expected: Technical & Engineering Documents
Predicted: Technical & Engineering Documents
Confidence: 0.391
Top 3 predictions:
  → 1. Technical & Engineering Documents: 0.391
    2. Strategic & Project Management Documents: 0.196
    3. Human Resources & Administrative Documents: 0.173
------------------------------------------------------------
Document 2: ✅ CORRECT
Text: Customer service feedback form documenting passenger complaints about train dela...
Expected: Passenger & Public-Facing Documents
Predicted: Passenger & Public-Facing Documents
Confidence: 0.497
Top 3 predictions:
  → 1. Passenger & Public-Facing Documents: 0.497
    2. Strategic & Project Management Documents: 0.131
    3. Human Resources & Administrative Documents: 0.129
------------------------------------------------------------
Document 

## 💾 Step 6: Save and Download the Model

Let's save the trained model and download it for future use.

In [ ]:
# Save the trained model
if training_success:
    print("💾 Saving the trained BERT model...")

    model_path = '/content/trained_metro_bert_model'

    try:
        classifier.save_model(model_path)
        print(f"✅ Model saved successfully to: {model_path}")

        # Check saved files and size
        print("\n📁 Saved files:")
        !ls -la /content/trained_metro_bert_model/

        print("\n📏 Model size:")
        !du -sh /content/trained_metro_bert_model/

        # Display training history
        if classifier.training_history:
            print("\n📈 Training History:")
            for key, value in classifier.training_history.items():
                if isinstance(value, float):
                    print(f"  • {key}: {value:.4f}")
                else:
                    print(f"  • {key}: {value}")

        model_saved = True

    except Exception as e:
        print(f"❌ Error saving model: {e}")
        model_saved = False

else:
    print("⚠️  Cannot save model - training was not successful.")
    model_saved = False

💾 Saving the trained BERT model...
✅ Model saved successfully to: /content/trained_metro_bert_model

📁 Saved files:
total 428656
drwxr-xr-x 2 root root      4096 Sep 16 10:42 .
drwxr-xr-x 1 root root      4096 Sep 16 10:42 ..
-rw-r--r-- 1 root root       949 Sep 16 10:55 config.json
-rw-r--r-- 1 root root      1456 Sep 16 10:55 metadata.json
-rw-r--r-- 1 root root 437970952 Sep 16 10:55 model.safetensors
-rw-r--r-- 1 root root       125 Sep 16 10:55 special_tokens_map.json
-rw-r--r-- 1 root root      1221 Sep 16 10:55 tokenizer_config.json
-rw-r--r-- 1 root root    711649 Sep 16 10:55 tokenizer.json
-rw-r--r-- 1 root root    231508 Sep 16 10:55 vocab.txt

📏 Model size:
419M	/content/trained_metro_bert_model/

📈 Training History:
  • training_time: 0:02:14.127986
  • train_loss: 1.0349
  • epochs_completed: 7.0000
  • global_step: 84
  • eval_loss: 0.9659
  • eval_accuracy: 0.7917
  • eval_f1: 0.7867
  • eval_precision: 0.7929
  • eval_recall: 0.7917
  • eval_runtime: 0.2637
  • eval_sa

In [ ]:
# Create downloadable zip file
if model_saved:
    print("📦 Creating downloadable model package...")

    import shutil
    from google.colab import files

    try:
        # Create zip file
        zip_path = '/content/trained_metro_bert_model'
        shutil.make_archive(zip_path, 'zip', '/content/trained_metro_bert_model')

        zip_file = zip_path + '.zip'
        print(f"✅ Model compressed to: {zip_file}")

        # Check zip file size
        !ls -lh /content/trained_metro_bert_model.zip

        print("\n📥 Starting download...")
        print("Note: Large file download may take several minutes")

        # Download the model
        files.download(zip_file)

        print("\n✅ Download initiated!")
        print("Check your Downloads folder for 'trained_metro_bert_model.zip'")

        # Create usage instructions
        usage_instructions = '''
# How to use your downloaded model:

1. Extract the zip file
2. Load the model in Python:

```python
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Load model and tokenizer
model_path = "path/to/extracted/model"
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Make predictions
text = "Your document text here"
inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
outputs = model(**inputs)
predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
```
        '''

        with open('/content/usage_instructions.txt', 'w') as f:
            f.write(usage_instructions)

        print("\n📝 Usage instructions created")
        files.download('/content/usage_instructions.txt')

    except Exception as e:
        print(f"❌ Error creating download package: {e}")

else:
    print("⚠️  Cannot download model - model was not saved successfully.")

📦 Creating downloadable model package...
✅ Model compressed to: /content/trained_metro_bert_model.zip
-rw-r--r-- 1 root root 388M Sep 16 10:55 /content/trained_metro_bert_model.zip

📥 Starting download...
Note: Large file download may take several minutes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Download initiated!
Check your Downloads folder for 'trained_metro_bert_model.zip'

📝 Usage instructions created


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 🔄 Step 7: Test Model Loading

Let's test loading the saved model to ensure it works correctly.

In [ ]:
# Test loading the saved model
if model_saved:
    print("🔄 Testing model loading...")

    try:
        # Create a new classifier instance
        test_classifier = EnhancedBERTDocumentClassifier()

        # Add load_model method
        def load_model(self, path):
            """Load trained model and tokenizer"""
            from pathlib import Path
            import json

            load_path = Path(path)

            if not load_path.exists():
                raise FileNotFoundError(f"Model path {load_path} does not exist")

            self.model = AutoModelForSequenceClassification.from_pretrained(load_path)
            self.tokenizer = AutoTokenizer.from_pretrained(load_path)

            metadata_path = load_path / 'metadata.json'
            if metadata_path.exists():
                with open(metadata_path, 'r') as f:
                    metadata = json.load(f)

                self.categories = metadata.get('categories', self.categories)
                self.label_to_id = metadata.get('label_to_id', self.label_to_id)
                self.id_to_label = metadata.get('id_to_label', self.id_to_label)
                self.training_history = metadata.get('training_history', {})

            logger.info(f"Model loaded from {load_path}")

        # Add method to test classifier
        test_classifier.load_model = lambda path: load_model(test_classifier, path)

        # Load the saved model
        test_classifier.load_model('/content/trained_metro_bert_model')

        print("✅ Model loaded successfully!")

        # Test prediction with loaded model
        test_text = "Maintenance inspection report for railway track alignment verification and signal system troubleshooting procedures"

        predictions = test_classifier.predict(test_text, return_all_scores=True)
        predicted_category = predictions[0][0]
        confidence = predictions[0][1]

        print(f"\n🧪 Test prediction with loaded model:")
        print(f"Text: {test_text}")
        print(f"Predicted: {predicted_category}")
        print(f"Confidence: {confidence:.3f}")

        print("\nTop 3 predictions:")
        for i, (category, conf) in enumerate(predictions[:3], 1):
            print(f"  {i}. {category}: {conf:.3f}")

        print("\n✅ Model loading test completed successfully!")

    except Exception as e:
        print(f"❌ Error testing model loading: {e}")

else:
    print("⚠️  Cannot test loading - model was not saved.")

🔄 Testing model loading...
✅ Model loaded successfully!
❌ Error testing model loading: Expected all tensors to be on the same device, but got index is on cuda:0, different from other tensors on cpu (when checking argument in method wrapper_CUDA__index_select)


## 🎉 Summary and Next Steps

Congratulations! You've successfully trained a BERT model for metro railway document classification.

In [ ]:
# Final summary and next steps
print("🎉 METRO RAILWAY DOCUMENT CLASSIFICATION - COMPLETE!")
print("=" * 60)

if training_success and model_saved:
    print("✅ Training: SUCCESSFUL")
    print("✅ Model Saved: YES")
    print("✅ Download: AVAILABLE")

    print(f"\n📊 Final Model Performance:")
    if 'eval_accuracy' in eval_results:
        print(f"  • Accuracy: {eval_results['eval_accuracy']:.1%}")
        print(f"  • F1-Score: {eval_results['eval_f1']:.3f}")

    print(f"\n📋 Model Details:")
    print(f"  • Categories: {len(classifier.categories)}")
    print(f"  • Training samples: {len(train_dataset)}")
    print(f"  • Validation samples: {len(val_dataset)}")
    print(f"  • Model size: ~438MB")

    print(f"\n🚀 Next Steps:")
    print("1. Download your trained model (files should be in Downloads folder)")
    print("2. Extract the ZIP file to use in your applications")
    print("3. Collect more real metro railway documents for better performance")
    print("4. Retrain with your specific documents using this same notebook")
    print("5. Deploy the model in your production environment")

    print(f"\n💡 Pro Tips:")
    print("• For better performance, train with 100+ documents per category")
    print("• Use GPU for faster inference in production")
    print("• Monitor performance and retrain periodically")
    print("• Consider ensemble methods for critical applications")

    print(f"\n📚 Resources:")
    print("• Hugging Face Transformers: https://huggingface.co/docs/transformers")
    print("• Model deployment guides: Check usage_instructions.txt")
    print("• Fine-tuning tips: https://huggingface.co/course")

else:
    print("⚠️  Some steps were not completed successfully.")
    print("\n🔧 Troubleshooting:")

    if not training_success:
        print("• Training failed - check GPU availability and memory")
        print("• Try reducing batch_size or switching to CPU")
        print("• Restart runtime if needed")

    if not model_saved:
        print("• Model saving failed - check disk space")
        print("• Try saving to a different location")

    print("\n📞 Support:")
    print("• Re-run failed cells after fixing issues")
    print("• Check error messages for specific guidance")
    print("• Consider using Colab Pro for better resources")

print(f"\n🙏 Thank you for using the Metro Railway Document Classifier!")
print("=" * 60)

🎉 METRO RAILWAY DOCUMENT CLASSIFICATION - COMPLETE!
✅ Training: SUCCESSFUL
✅ Model Saved: YES
✅ Download: AVAILABLE

📊 Final Model Performance:
  • Accuracy: 79.2%
  • F1-Score: 0.787

📋 Model Details:
  • Categories: 6
  • Training samples: 96
  • Validation samples: 24
  • Model size: ~438MB

🚀 Next Steps:
1. Download your trained model (files should be in Downloads folder)
2. Extract the ZIP file to use in your applications
3. Collect more real metro railway documents for better performance
4. Retrain with your specific documents using this same notebook
5. Deploy the model in your production environment

💡 Pro Tips:
• For better performance, train with 100+ documents per category
• Use GPU for faster inference in production
• Monitor performance and retrain periodically
• Consider ensemble methods for critical applications

📚 Resources:
• Hugging Face Transformers: https://huggingface.co/docs/transformers
• Model deployment guides: Check usage_instructions.txt
• Fine-tuning tips: h